# Transformation

Fits the LLM transformation function $\hat{f}(x) = \mathbb{E}[\text{transformed} \mid \text{original}=x]$ via Nadaraya-Watson kernel regression with leave-one-out cross-validated bandwidth. It writes one estimated function per `(dataset, task, model, topic, quantification_method)` in `outputs/transformation/transformation__*.json`, which is later used by `src/simulation.jl`.

In [1]:
import os
os.chdir("../")

In [ ]:
import glob
import json

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src import utils

sns.set_theme(context='paper', style='ticks', font_scale=1)

os.environ['PATH'] = f"{os.path.expanduser('~/.TinyTeX/bin/x86_64-linux')}:{os.environ['PATH']}"

In [ ]:
name = "transformation"
width_pt = 469

quantification_method = "centroid"

predict_dir = "outputs/predict_opinion"
out_dir = "outputs/transformation"
os.makedirs(out_dir, exist_ok=True)

n_grid = 4001
x_grid = np.linspace(0.0, 1.0, n_grid)

h_grid = np.geomspace(2e-3, 3e-1, 50)

## Helpers

In [ ]:
def silverman_bandwidth(o):
    return 1.06 * np.std(o) * len(o) ** (-1 / 5)


def nadaraya_watson(o, t, x_grid, h, weights=None):
    o = np.asarray(o, dtype=float)
    t = np.asarray(t, dtype=float)
    w = np.ones_like(o) if weights is None else np.asarray(weights, dtype=float)
    diffs = x_grid[:, None] - o[None, :]
    K = np.exp(-(diffs ** 2) / (2.0 * h * h))
    num = K @ (w * t)
    den = K @ w
    out = np.empty(len(x_grid))
    valid = den > 1e-300
    out[valid] = num[valid] / den[valid]
    if (~valid).any():
        nearest = np.argmin(np.abs(o[None, :] - x_grid[~valid, None]), axis=1)
        out[~valid] = t[nearest]
    return out


def loo_cv_mse(o, t, h_grid, weights=None):
    o = np.asarray(o, dtype=float)
    t = np.asarray(t, dtype=float)
    w = np.ones_like(o) if weights is None else np.asarray(weights, dtype=float)
    diffs2 = (o[:, None] - o[None, :]) ** 2
    t_bar = float((w * t).sum() / w.sum())
    mse = np.empty(len(h_grid))
    for k, h in enumerate(h_grid):
        K = np.exp(-diffs2 / (2.0 * h * h))
        num = K @ (w * t) - w * t
        den = K @ w - w
        valid = den > 1e-12
        f_loo = np.where(valid, num / np.where(valid, den, 1.0), t_bar)
        mse[k] = np.mean((t - f_loo) ** 2)
    return mse


def load_and_aggregate(filepath):
    df = pd.read_csv(filepath, sep="\t", dtype=str, quoting=3, on_bad_lines='warn')
    df['confidence_original'] = df['confidence_original'].astype(float)
    df['confidence_transformed'] = df['confidence_transformed'].astype(float)
    df = df[df['prediction_original'] == df['prediction_transformed']]
    agg = df.groupby('sentence_id').agg(
        confidence_original=('confidence_original', 'first'),
        confidence_transformed=('confidence_transformed', 'mean'),
        n_surviving=('confidence_transformed', 'size'),
    ).reset_index()
    return agg

## Bandwidth selection and grid precomputation

In [6]:
predict_pattern = (
    f"{predict_dir}/predict_opinion__dataset=*__task=*__model=*__topic=*"
    f"__quantification_method={quantification_method}.tsv"
)
predict_files = sorted(glob.glob(predict_pattern))
print(f"Found {len(predict_files)} predict_opinion files")

fits = {}
for path in predict_files:
    basename = os.path.basename(path).replace(".tsv", "")
    parts = dict(p.split("=", 1) for p in basename.split("__")[1:])
    ds, tk, md, tp, qm = parts['dataset'], parts['task'], parts['model'], parts['topic'], parts['quantification_method']

    agg = load_and_aggregate(path)
    o = agg['confidence_original'].to_numpy()
    t = agg['confidence_transformed'].to_numpy()
    n_surv = agg['n_surviving'].to_numpy().astype(float)
    if len(o) < 5:
        print(f"  skip (too few points: {len(o)}): {basename}")
        continue

    cv_mse = loo_cv_mse(o, t, h_grid, weights=n_surv)
    h_star = float(h_grid[int(np.argmin(cv_mse))])
    f_grid = nadaraya_watson(o, t, x_grid, h_star, weights=n_surv)

    out_path = (
        f"{out_dir}/transformation__dataset={ds}__task={tk}"
        f"__model={md}__topic={tp}__quantification_method={qm}.json"
    )
    with open(out_path, 'w') as f:
        json.dump({
            'dataset': ds, 'task': tk, 'model': md, 'topic': tp,
            'quantification_method': qm,
            'n_statements': int(len(o)),
            'n_surviving_total': int(n_surv.sum()),
            'n_surviving_mean': float(n_surv.mean()),
            'n_surviving_median': float(np.median(n_surv)),
            'weighting': 'n_surviving_rewrites',
            'bandwidth': h_star,
            'silverman_bandwidth': float(silverman_bandwidth(o)),
            'n_grid': int(n_grid),
            'f_grid': f_grid.tolist(),
            'cv_curve': {'h': h_grid.tolist(), 'mse': cv_mse.tolist()},
        }, f)

    fits[(ds, tk, md, tp, qm)] = {
        'o': o, 't': t, 'n_surv': n_surv,
        'h_star': h_star,
        'h_silverman': float(silverman_bandwidth(o)),
        'f_grid': f_grid,
        'cv_mse': cv_mse,
    }
    print(f"  {basename}: n={len(o)} sentences ({int(n_surv.sum())} surviving rewrites, "
          f"mean {n_surv.mean():.2f}/sentence), h*={h_star:.4f} (Silverman={silverman_bandwidth(o):.4f})")

print(f"Wrote {len(fits)} transformation files to {out_dir}/")


Found 56 predict_opinion files
  predict_opinion__dataset=semeval__task=improvement__model=Qwen_Qwen3-8B__topic=abortion__quantification_method=centroid: n=294 sentences (4303 surviving rewrites, mean 14.64/sentence), h*=0.0258 (Silverman=0.1095)
  predict_opinion__dataset=semeval__task=improvement__model=Qwen_Qwen3-8B__topic=acknowledging_climate_change__quantification_method=centroid: n=51 sentences (714 surviving rewrites, mean 14.00/sentence), h*=0.0527 (Silverman=0.1868)
  predict_opinion__dataset=semeval__task=improvement__model=Qwen_Qwen3-8B__topic=atheism__quantification_method=centroid: n=239 sentences (3510 surviving rewrites, mean 14.69/sentence), h*=0.0190 (Silverman=0.1243)
  predict_opinion__dataset=semeval__task=improvement__model=Qwen_Qwen3-8B__topic=donald_trump__quantification_method=centroid: n=254 sentences (3711 surviving rewrites, mean 14.61/sentence), h*=0.0210 (Silverman=0.1404)
  predict_opinion__dataset=semeval__task=improvement__model=Qwen_Qwen3-8B__topic=fem

## LLM transformation function

In [8]:
utils.latexify()

for (ds, tk, md, tp, qm), fit in fits.items():
    fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    sns.scatterplot(x=fit['o'], y=fit['t'], color='silver', s=20, alpha=0.3,
                    edgecolor='none', ax=ax)
    ax.axline((0, 0), slope=1, color='gray', linestyle='--', linewidth=1, zorder=0)
    sns.lineplot(x=x_grid, y=fit['f_grid'], color='magenta', ax=ax)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel(r"Original opinion (human)")
    ax.set_ylabel(r"Transformed opinion (LLM)")

    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(
        f"figures/{name}__function__model={md}__dataset={ds}"
        f"__topic={tp}.pdf",
        dpi=300,
    )
    plt.close()

## Bandwidth selection diagnostic

In [7]:
utils.latexify()

for (ds, tk, md, tp, qm), fit in fits.items():
    fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    sns.lineplot(x=h_grid, y=fit['cv_mse'], color='black', ax=ax)
    ax.axvline(fit['h_silverman'], color='gray', linestyle='--', linewidth=1,
               label='Silverman')
    ax.scatter([fit['h_star']], [fit['cv_mse'].min()], color='magenta', zorder=5,
               label=r'$h^\star$')

    ax.set_xscale('log')
    ax.set_xlabel(r"Bandwidth $h$")
    ax.set_ylabel(r"LOO-CV MSE")
    ax.legend(loc='best', framealpha=0.9)

    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(
        f"figures/{name}__cv_curve__model={md}__dataset={ds}"
        f"__task={tk}__topic={tp}__quantification_method={qm}.pdf",
        dpi=300,
    )
    plt.close()